# Strategy Parameter Optimization - Independent Parameters per Asset

This notebook uses **grid search over the parameter space** to independently find the optimal strategy and parameters for each asset.

## Optimization Method
- **Grid Search**: Exhaustive search over all parameter combinations
- **Train/Test Split**: 7:3 (700 days training, 300 days testing)
- **Evaluation Metric**: Composite score = 0.3 x Train Sharpe + 0.3 x Test Sharpe + 0.2 x Test Return + 0.2 x (1 - |Drawdown|/50)
- **Independent Optimization**: Each asset is optimized independently for the best strategy type and parameters

## Asset Groups (Based on Return/MaxDD Ratio)
- **Trend/Momentum Strategy**: series_1, 3, 4, 6, 9 (Return/MaxDD >= 1.1)
- **Mean Reversion Strategy**: series_2, 7, 8, 10 (Return/MaxDD < 1.1)
- **Short/Defensive Strategy**: series_5 (Total return < 0)

Please run data_analysis.ipynb first.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import product
from typing import Dict, List
import warnings
import json
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Path setup
DATA_PATH = Path('../BT396_Version_0.1.3/DATA/PART2')
RESULTS_PATH = Path('./optimization_results')
RESULTS_PATH.mkdir(exist_ok=True)

print(f'Data path: {DATA_PATH.absolute()}')
print(f'Results path: {RESULTS_PATH.absolute()}')

Data path: c:\Users\14671\Desktop\Optimization\..\BT396_Version_0.1.3\DATA\PART2
Results path: c:\Users\14671\Desktop\Optimization\optimization_results


## 1. Data Loading and Splitting

In [2]:
def load_data() -> Dict[str, pd.DataFrame]:
    """Load all 10 series data"""
    data = {}
    for i in range(1, 11):
        filename = f'{i:02d}.csv'
        filepath = DATA_PATH / filename
        df = pd.read_csv(filepath)
        
        date_col = 'Index' if 'Index' in df.columns else 'Date'
        df[date_col] = pd.to_datetime(df[date_col])
        df.set_index(date_col, inplace=True)
        df.sort_index(inplace=True)
        
        data[f'series_{i}'] = df
    
    return data

# Load data
data = load_data()
print(f'Loaded {len(data)} series')

# 7:3 split
TOTAL_DAYS = len(data['series_1'])
TRAIN_END = int(TOTAL_DAYS * 0.7)  # 700 days

print(f'\nData period: {data["series_1"].index.min().date()} to {data["series_1"].index.max().date()}')
print(f'Total days: {TOTAL_DAYS}')
print(f'Training set: day 1-{TRAIN_END} ({TRAIN_END} days, {TRAIN_END/TOTAL_DAYS*100:.0f}%)')
print(f'Test set: day {TRAIN_END+1}-{TOTAL_DAYS} ({TOTAL_DAYS-TRAIN_END} days, {(TOTAL_DAYS-TRAIN_END)/TOTAL_DAYS*100:.0f}%)')

# Asset groups
TREND_SERIES = ['series_1', 'series_3', 'series_4', 'series_6', 'series_9']
MR_SERIES = ['series_2', 'series_7', 'series_8', 'series_10']
SHORT_SERIES = ['series_5']

print(f'\nTrend/momentum assets: {TREND_SERIES}')
print(f'Mean reversion assets: {MR_SERIES}')
print(f'Short/defensive assets: {SHORT_SERIES}')

Loaded 10 series

Data period: 2072-09-03 to 2075-05-30
Total days: 1000
Training set: day 1-700 (700 days, 70%)
Test set: day 701-1000 (300 days, 30%)

Trend/momentum assets: ['series_1', 'series_3', 'series_4', 'series_6', 'series_9']
Mean reversion assets: ['series_2', 'series_7', 'series_8', 'series_10']
Short/defensive assets: ['series_5']


## 2. Backtest Engine

In [3]:
class SimpleBacktest:
    """Simplified backtest engine for fast parameter optimization"""
    
    def __init__(self, df: pd.DataFrame, initial_cash: float = 100000):
        self.df = df.copy()
        self.initial_cash = initial_cash
        self.cash = initial_cash
        self.position = 0
        self.equity_curve = []
        self.trades = []
    
    def run(self, signals: pd.Series) -> Dict:
        """Run backtest"""
        self.cash = self.initial_cash
        self.position = 0
        self.equity_curve = []
        self.trades = []
        
        close = self.df['Close'].values
        
        for i in range(len(self.df)):
            price = close[i]
            signal = signals.iloc[i] if i < len(signals) else 0
            
            if signal == 1 and self.position <= 0:  # Buy signal
                if self.position < 0:
                    self.cash += self.position * price
                shares = int(self.cash * 0.95 / price)
                if shares > 0:
                    self.position = shares
                    self.cash -= shares * price
                    self.trades.append({'type': 'buy', 'price': price, 'day': i})
            
            elif signal == -1 and self.position > 0:  # Sell signal
                self.cash += self.position * price
                self.trades.append({'type': 'sell', 'price': price, 'day': i})
                self.position = 0
            
            elif signal == -2 and self.position >= 0:  # Short signal
                if self.position > 0:
                    self.cash += self.position * price
                shares = int(self.cash * 0.95 / price)
                if shares > 0:
                    self.position = -shares
                    self.cash += shares * price
                    self.trades.append({'type': 'short', 'price': price, 'day': i})
            
            equity = self.cash + self.position * price
            self.equity_curve.append(equity)
        
        # Final liquidation
        final_price = close[-1]
        if self.position != 0:
            self.cash += self.position * final_price
            self.position = 0
        
        return self._calculate_metrics()
    
    def _calculate_metrics(self) -> Dict:
        """Calculate backtest metrics"""
        equity = np.array(self.equity_curve)
        returns = np.diff(equity) / equity[:-1]
        
        total_return = (equity[-1] / self.initial_cash - 1) * 100
        sharpe = np.mean(returns) / np.std(returns) * np.sqrt(252) if np.std(returns) > 0 else 0
        
        cummax = np.maximum.accumulate(equity)
        drawdown = (equity - cummax) / cummax
        max_drawdown = np.min(drawdown) * 100
        
        n_trades = len([t for t in self.trades if t['type'] in ['buy', 'short']])
        
        return {
            'total_return': total_return,
            'sharpe': sharpe,
            'max_drawdown': max_drawdown,
            'n_trades': n_trades,
            'final_equity': equity[-1]
        }

print('Backtest engine defined')

Backtest engine defined


---
# Part 1: Trend/Momentum Strategy Parameter Optimization

**Independently find optimal parameters** for each trend asset, searching across 4 strategy variants.

## Strategy Variants
1. **Dual Moving Average Crossover + ATR Stop Loss** (ma_crossover)
2. **Momentum Breakout** (breakout)
3. **MACD** (macd)
4. **Above Moving Average Confirmation** (above_ma)

## 3. Trend Strategy Signal Generation Functions

In [4]:
# ============================================================
# Strategy 1: Dual Moving Average Crossover + ATR Stop Loss
# ============================================================
def generate_trend_signals_ma(df: pd.DataFrame, fast_period: int, slow_period: int,
                               atr_period: int, atr_mult: float) -> pd.Series:
    """Dual MA crossover + ATR trailing stop loss"""
    close = df['Close']
    high = df['High']
    low = df['Low']
    
    fast_ma = close.rolling(fast_period).mean()
    slow_ma = close.rolling(slow_period).mean()
    
    # ATR calculation
    tr1 = high - low
    tr2 = abs(high - close.shift(1))
    tr3 = abs(low - close.shift(1))
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(atr_period).mean()
    
    signals = pd.Series(0, index=df.index)
    position = 0
    stop_loss = 0
    
    for i in range(slow_period, len(df)):
        if position == 0:
            if fast_ma.iloc[i] > slow_ma.iloc[i] and fast_ma.iloc[i-1] <= slow_ma.iloc[i-1]:
                signals.iloc[i] = 1
                position = 1
                stop_loss = close.iloc[i] - atr_mult * atr.iloc[i]
        else:
            if fast_ma.iloc[i] < slow_ma.iloc[i] and fast_ma.iloc[i-1] >= slow_ma.iloc[i-1]:
                signals.iloc[i] = -1
                position = 0
            elif close.iloc[i] < stop_loss:
                signals.iloc[i] = -1
                position = 0
            else:
                new_stop = close.iloc[i] - atr_mult * atr.iloc[i]
                stop_loss = max(stop_loss, new_stop)
    
    return signals


# ============================================================
# Strategy 2: Momentum Breakout
# ============================================================
def generate_trend_signals_breakout(df: pd.DataFrame, lookback: int,
                                     hold_days: int) -> pd.Series:
    """Buy on N-day high breakout, sell after holding period or on N-day low breakdown"""
    close = df['Close']
    
    rolling_high = close.rolling(lookback).max()
    rolling_low = close.rolling(lookback).min()
    
    signals = pd.Series(0, index=df.index)
    position = 0
    entry_day = 0
    
    for i in range(lookback, len(df)):
        if position == 0:
            if close.iloc[i] >= rolling_high.iloc[i-1]:
                signals.iloc[i] = 1
                position = 1
                entry_day = i
        else:
            if (i - entry_day >= hold_days) or (close.iloc[i] <= rolling_low.iloc[i-1]):
                signals.iloc[i] = -1
                position = 0
    
    return signals


# ============================================================
# Strategy 3: MACD Trend Strategy
# ============================================================
def generate_trend_signals_macd(df: pd.DataFrame, fast_period: int,
                                 slow_period: int, signal_period: int) -> pd.Series:
    """MACD golden cross buy, death cross sell"""
    close = df['Close']
    
    ema_fast = close.ewm(span=fast_period, adjust=False).mean()
    ema_slow = close.ewm(span=slow_period, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal_period, adjust=False).mean()
    
    signals = pd.Series(0, index=df.index)
    position = 0
    
    for i in range(slow_period + signal_period, len(df)):
        if position == 0:
            if macd_line.iloc[i] > signal_line.iloc[i] and macd_line.iloc[i-1] <= signal_line.iloc[i-1]:
                signals.iloc[i] = 1
                position = 1
        else:
            if macd_line.iloc[i] < signal_line.iloc[i] and macd_line.iloc[i-1] >= signal_line.iloc[i-1]:
                signals.iloc[i] = -1
                position = 0
    
    return signals


# ============================================================
# Strategy 4: Trend Following (Price Above Moving Average)
# ============================================================
def generate_trend_signals_above_ma(df: pd.DataFrame, ma_period: int,
                                     confirm_days: int) -> pd.Series:
    """Buy when price stays above MA for N consecutive days, sell when price drops below MA"""
    close = df['Close']
    ma = close.rolling(ma_period).mean()
    
    signals = pd.Series(0, index=df.index)
    position = 0
    
    for i in range(ma_period + confirm_days, len(df)):
        if position == 0:
            above_count = sum(close.iloc[i-confirm_days+1:i+1] > ma.iloc[i-confirm_days+1:i+1])
            if above_count == confirm_days:
                signals.iloc[i] = 1
                position = 1
        else:
            if close.iloc[i] < ma.iloc[i]:
                signals.iloc[i] = -1
                position = 0
    
    return signals

print('Trend strategy signal functions defined: ma_crossover, breakout, macd, above_ma')

Trend strategy signal functions defined: ma_crossover, breakout, macd, above_ma


## 4. Trend Strategy Parameter Grid Definition

In [5]:
# ============================================================
# Parameter grid definition
# ============================================================

# Strategy 1: Dual MA Crossover + ATR Stop Loss
MA_PARAM_GRID = {
    'fast_period': [3, 5, 8, 10, 15, 20],      # 6 values
    'slow_period': [15, 20, 25, 30, 40, 50, 60], # 7 values
    'atr_period': [10, 14, 20],                 # 3 values
    'atr_mult': [1.5, 2.0, 2.5, 3.0, 3.5]       # 5 values
}

# Strategy 2: Momentum Breakout
BREAKOUT_PARAM_GRID = {
    'lookback': [5, 10, 15, 20, 25, 30, 40],    # 7 values
    'hold_days': [5, 10, 15, 20, 30, 40, 50]    # 7 values
}

# Strategy 3: MACD
MACD_PARAM_GRID = {
    'fast_period': [8, 10, 12, 15],             # 4 values
    'slow_period': [20, 26, 30, 35],            # 4 values
    'signal_period': [5, 7, 9, 12]              # 4 values
}

# Strategy 4: Above MA Confirmation
ABOVE_MA_PARAM_GRID = {
    'ma_period': [10, 15, 20, 30, 40, 50, 60],  # 7 values
    'confirm_days': [1, 2, 3, 5, 7, 10]         # 6 values
}

print('=== Trend Strategy Parameter Grid ===')
print(f'\nStrategy 1 - Dual MA Crossover + ATR:')
for k, v in MA_PARAM_GRID.items():
    print(f'  {k}: {v}')
print(f'  Estimated combinations: ~441 (after filtering fast>=slow)')

print(f'\nStrategy 2 - Momentum Breakout:')
for k, v in BREAKOUT_PARAM_GRID.items():
    print(f'  {k}: {v}')
print(f'  Combinations: {len(BREAKOUT_PARAM_GRID["lookback"]) * len(BREAKOUT_PARAM_GRID["hold_days"])}')

print(f'\nStrategy 3 - MACD:')
for k, v in MACD_PARAM_GRID.items():
    print(f'  {k}: {v}')
print(f'  Estimated combinations: ~48 (after filtering fast>=slow)')

print(f'\nStrategy 4 - Above MA Confirmation:')
for k, v in ABOVE_MA_PARAM_GRID.items():
    print(f'  {k}: {v}')
print(f'  Combinations: {len(ABOVE_MA_PARAM_GRID["ma_period"]) * len(ABOVE_MA_PARAM_GRID["confirm_days"])}')

print(f'\nEstimated total parameter combinations per asset: ~580')

=== Trend Strategy Parameter Grid ===

Strategy 1 - Dual MA Crossover + ATR:
  fast_period: [3, 5, 8, 10, 15, 20]
  slow_period: [15, 20, 25, 30, 40, 50, 60]
  atr_period: [10, 14, 20]
  atr_mult: [1.5, 2.0, 2.5, 3.0, 3.5]
  Estimated combinations: ~441 (after filtering fast>=slow)

Strategy 2 - Momentum Breakout:
  lookback: [5, 10, 15, 20, 25, 30, 40]
  hold_days: [5, 10, 15, 20, 30, 40, 50]
  Combinations: 49

Strategy 3 - MACD:
  fast_period: [8, 10, 12, 15]
  slow_period: [20, 26, 30, 35]
  signal_period: [5, 7, 9, 12]
  Estimated combinations: ~48 (after filtering fast>=slow)

Strategy 4 - Above MA Confirmation:
  ma_period: [10, 15, 20, 30, 40, 50, 60]
  confirm_days: [1, 2, 3, 5, 7, 10]
  Combinations: 42

Estimated total parameter combinations per asset: ~580


## 5. Single-Asset Trend Strategy Optimization Function

In [6]:
def optimize_trend_single_asset(df: pd.DataFrame, series_name: str, train_end: int) -> pd.DataFrame:
    """
    Search all trend strategy types and parameter combinations for a single asset,
    return results sorted by score.
    """
    results = []
    
    # ============================================================
    # Strategy 1: Dual MA Crossover + ATR Stop Loss
    # ============================================================
    for fast, slow, atr_p, atr_m in product(
        MA_PARAM_GRID['fast_period'],
        MA_PARAM_GRID['slow_period'],
        MA_PARAM_GRID['atr_period'],
        MA_PARAM_GRID['atr_mult']
    ):
        if fast >= slow:
            continue
        
        try:
            signals = generate_trend_signals_ma(df, fast, slow, atr_p, atr_m)
            
            bt_train = SimpleBacktest(df.iloc[:train_end])
            train_result = bt_train.run(signals.iloc[:train_end])
            
            bt_valid = SimpleBacktest(df.iloc[train_end:])
            valid_result = bt_valid.run(signals.iloc[train_end:])
            
            score = (
                0.3 * np.clip(train_result['sharpe'], -2, 2) / 2 +
                0.3 * np.clip(valid_result['sharpe'], -2, 2) / 2 +
                0.2 * np.clip(valid_result['total_return'], -50, 100) / 100 +
                0.2 * (1 - abs(np.clip(valid_result['max_drawdown'], -50, 0)) / 50)
            )
            
            results.append({
                'strategy': 'ma_crossover',
                'params': f'fast={fast},slow={slow},atr_p={atr_p},atr_m={atr_m}',
                'train_return': train_result['total_return'],
                'train_sharpe': train_result['sharpe'],
                'valid_return': valid_result['total_return'],
                'valid_sharpe': valid_result['sharpe'],
                'valid_maxdd': valid_result['max_drawdown'],
                'score': score
            })
        except:
            continue
    
    # ============================================================
    # Strategy 2: Momentum Breakout
    # ============================================================
    for lookback, hold_days in product(
        BREAKOUT_PARAM_GRID['lookback'],
        BREAKOUT_PARAM_GRID['hold_days']
    ):
        try:
            signals = generate_trend_signals_breakout(df, lookback, hold_days)
            
            bt_train = SimpleBacktest(df.iloc[:train_end])
            train_result = bt_train.run(signals.iloc[:train_end])
            
            bt_valid = SimpleBacktest(df.iloc[train_end:])
            valid_result = bt_valid.run(signals.iloc[train_end:])
            
            score = (
                0.3 * np.clip(train_result['sharpe'], -2, 2) / 2 +
                0.3 * np.clip(valid_result['sharpe'], -2, 2) / 2 +
                0.2 * np.clip(valid_result['total_return'], -50, 100) / 100 +
                0.2 * (1 - abs(np.clip(valid_result['max_drawdown'], -50, 0)) / 50)
            )
            
            results.append({
                'strategy': 'breakout',
                'params': f'lookback={lookback},hold_days={hold_days}',
                'train_return': train_result['total_return'],
                'train_sharpe': train_result['sharpe'],
                'valid_return': valid_result['total_return'],
                'valid_sharpe': valid_result['sharpe'],
                'valid_maxdd': valid_result['max_drawdown'],
                'score': score
            })
        except:
            continue
    
    # ============================================================
    # Strategy 3: MACD
    # ============================================================
    for fast, slow, signal in product(
        MACD_PARAM_GRID['fast_period'],
        MACD_PARAM_GRID['slow_period'],
        MACD_PARAM_GRID['signal_period']
    ):
        if fast >= slow:
            continue
        
        try:
            signals = generate_trend_signals_macd(df, fast, slow, signal)
            
            bt_train = SimpleBacktest(df.iloc[:train_end])
            train_result = bt_train.run(signals.iloc[:train_end])
            
            bt_valid = SimpleBacktest(df.iloc[train_end:])
            valid_result = bt_valid.run(signals.iloc[train_end:])
            
            score = (
                0.3 * np.clip(train_result['sharpe'], -2, 2) / 2 +
                0.3 * np.clip(valid_result['sharpe'], -2, 2) / 2 +
                0.2 * np.clip(valid_result['total_return'], -50, 100) / 100 +
                0.2 * (1 - abs(np.clip(valid_result['max_drawdown'], -50, 0)) / 50)
            )
            
            results.append({
                'strategy': 'macd',
                'params': f'fast={fast},slow={slow},signal={signal}',
                'train_return': train_result['total_return'],
                'train_sharpe': train_result['sharpe'],
                'valid_return': valid_result['total_return'],
                'valid_sharpe': valid_result['sharpe'],
                'valid_maxdd': valid_result['max_drawdown'],
                'score': score
            })
        except:
            continue
    
    # ============================================================
    # Strategy 4: Above MA Confirmation
    # ============================================================
    for ma_period, confirm_days in product(
        ABOVE_MA_PARAM_GRID['ma_period'],
        ABOVE_MA_PARAM_GRID['confirm_days']
    ):
        try:
            signals = generate_trend_signals_above_ma(df, ma_period, confirm_days)
            
            bt_train = SimpleBacktest(df.iloc[:train_end])
            train_result = bt_train.run(signals.iloc[:train_end])
            
            bt_valid = SimpleBacktest(df.iloc[train_end:])
            valid_result = bt_valid.run(signals.iloc[train_end:])
            
            score = (
                0.3 * np.clip(train_result['sharpe'], -2, 2) / 2 +
                0.3 * np.clip(valid_result['sharpe'], -2, 2) / 2 +
                0.2 * np.clip(valid_result['total_return'], -50, 100) / 100 +
                0.2 * (1 - abs(np.clip(valid_result['max_drawdown'], -50, 0)) / 50)
            )
            
            results.append({
                'strategy': 'above_ma',
                'params': f'ma_period={ma_period},confirm_days={confirm_days}',
                'train_return': train_result['total_return'],
                'train_sharpe': train_result['sharpe'],
                'valid_return': valid_result['total_return'],
                'valid_sharpe': valid_result['sharpe'],
                'valid_maxdd': valid_result['max_drawdown'],
                'score': score
            })
        except:
            continue
    
    # Sort and return
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('score', ascending=False)
    
    return results_df

print('Single-asset trend strategy optimization function defined')

Single-asset trend strategy optimization function defined


## 6. Run Trend Strategy Parameter Optimization

In [7]:
print('='*80)
print('Trend/Momentum Strategy Parameter Optimization - Independent per Asset')
print('='*80)

trend_optimal_params = {}
trend_summary = []

for series_name in TREND_SERIES:
    print(f'\n{"="*60}')
    print(f'Optimizing {series_name}...')
    print(f'{"="*60}')
    
    df = data[series_name]
    results_df = optimize_trend_single_asset(df, series_name, TRAIN_END)
    
    # Save full results
    results_df.to_csv(RESULTS_PATH / f'trend_{series_name}_all_params.csv', index=False)
    
    # Get best result
    best = results_df.iloc[0]
    
    print(f'\nSearched {len(results_df)} parameter combinations')
    print(f'\n{series_name} optimal strategy:')
    print(f'  Strategy type: {best["strategy"]}')
    print(f'  Parameters: {best["params"]}')
    print(f'  Training return: {best["train_return"]:.2f}%')
    print(f'  Test return: {best["valid_return"]:.2f}%')
    print(f'  Test Sharpe: {best["valid_sharpe"]:.2f}')
    print(f'  Test MaxDD: {best["valid_maxdd"]:.2f}%')
    print(f'  Composite score: {best["score"]:.4f}')
    
    # Top 5
    print(f'\nTop 5 parameter combinations:')
    display(results_df.head())
    
    # Save optimal params
    trend_optimal_params[series_name] = {
        'strategy': best['strategy'],
        'params': best['params'],
        'train_return': float(best['train_return']),
        'valid_return': float(best['valid_return']),
        'valid_sharpe': float(best['valid_sharpe']),
        'valid_maxdd': float(best['valid_maxdd'])
    }
    
    trend_summary.append({
        'series': series_name,
        'strategy': best['strategy'],
        'params': best['params'],
        'train_return': best['train_return'],
        'valid_return': best['valid_return'],
        'valid_sharpe': best['valid_sharpe'],
        'score': best['score']
    })

Trend/Momentum Strategy Parameter Optimization - Independent per Asset

Optimizing series_1...

Searched 740 parameter combinations

series_1 optimal strategy:
  Strategy type: above_ma
  Parameters: ma_period=10,confirm_days=7
  Training return: 39.10%
  Test return: 12.06%
  Test Sharpe: 0.69
  Test MaxDD: -11.34%
  Composite score: 0.4294

Top 5 parameter combinations:


,strategy,params,train_return,train_sharpe,valid_return,valid_sharpe,valid_maxdd,score
702,above_ma,"ma_period=10,confirm_days=7",39.09546,0.985263,12.06011,0.685916,-11.341147,0.429433
630,breakout,"lookback=40,hold_days=20",22.84108,0.564800,12.81796,0.955795,-10.769899,0.410646
700,above_ma,"ma_period=10,confirm_days=3",26.07185,0.601808,17.67691,0.793205,-10.204893,0.403786
703,above_ma,"ma_period=10,confirm_days=10",17.65048,0.605726,10.84779,0.819508,-11.271347,0.390395
699,above_ma,"ma_period=10,confirm_days=2",25.54576,0.569771,18.65581,0.800920,-14.951826,0.383108



Optimizing series_3...

Searched 740 parameter combinations

series_3 optimal strategy:
  Strategy type: ma_crossover
  Parameters: fast=20,slow=40,atr_p=10,atr_m=2.5
  Training return: 11.49%
  Test return: 12.33%
  Test Sharpe: 1.82
  Test MaxDD: -5.18%
  Composite score: 0.6182

Top 5 parameter combinations:


,strategy,params,train_return,train_sharpe,valid_return,valid_sharpe,valid_maxdd,score
542,ma_crossover,"fast=20,slow=40,atr_p=10,atr_m=2.5",11.494946,0.940802,12.332610,1.821230,-5.183895,0.618235
547,ma_crossover,"fast=20,slow=40,atr_p=14,atr_m=2.5",11.342264,0.918570,12.332610,1.821230,-5.183895,0.614900
552,ma_crossover,"fast=20,slow=40,atr_p=20,atr_m=2.5",11.342264,0.918570,12.332610,1.821230,-5.183895,0.614900
424,ma_crossover,"fast=15,slow=20,atr_p=10,atr_m=3.5",6.488193,0.436237,16.567946,2.056016,-3.073409,0.586278
548,ma_crossover,"fast=20,slow=40,atr_p=14,atr_m=3.0",13.775437,0.930810,11.217154,1.648468,-6.125399,0.584824



Optimizing series_4...

Searched 740 parameter combinations

series_4 optimal strategy:
  Strategy type: ma_crossover
  Parameters: fast=15,slow=30,atr_p=14,atr_m=1.5
  Training return: 22.13%
  Test return: 46.76%
  Test Sharpe: 1.94
  Test MaxDD: -9.26%
  Composite score: 0.6148

Top 5 parameter combinations:


,strategy,params,train_return,train_sharpe,valid_return,valid_sharpe,valid_maxdd,score
455,ma_crossover,"fast=15,slow=30,atr_p=14,atr_m=1.5",22.12609,0.448245,46.76228,1.940744,-9.258340,0.614839
460,ma_crossover,"fast=15,slow=30,atr_p=20,atr_m=1.5",32.66397,0.607382,43.61529,1.751390,-9.256812,0.604019
425,ma_crossover,"fast=15,slow=20,atr_p=14,atr_m=1.5",37.80802,0.611780,42.89337,1.763628,-9.903190,0.602485
430,ma_crossover,"fast=15,slow=20,atr_p=20,atr_m=1.5",41.30226,0.657880,41.48552,1.697991,-9.903728,0.596737
530,ma_crossover,"fast=20,slow=30,atr_p=14,atr_m=1.5",15.94237,0.339647,40.46770,1.772986,-6.056539,0.573604



Optimizing series_6...

Searched 740 parameter combinations

series_6 optimal strategy:
  Strategy type: breakout
  Parameters: lookback=25,hold_days=20
  Training return: 26.02%
  Test return: 23.44%
  Test Sharpe: 1.29
  Test MaxDD: -6.80%
  Composite score: 0.4850

Top 5 parameter combinations:


,strategy,params,train_return,train_sharpe,valid_return,valid_sharpe,valid_maxdd,score
616,breakout,"lookback=25,hold_days=20",26.021436,0.480191,23.440454,1.288207,-6.795314,0.484959
585,breakout,"lookback=5,hold_days=5",74.755187,1.058484,14.662224,0.741490,-9.089210,0.462964
665,macd,"fast=10,slow=35,signal=12",93.575968,1.185318,7.675227,0.399117,-13.391597,0.399449
693,macd,"fast=15,slow=30,signal=12",75.388938,1.017250,9.427898,0.464870,-12.323796,0.391879
692,macd,"fast=15,slow=30,signal=9",91.302905,1.162905,6.681448,0.362613,-13.391191,0.388626



Optimizing series_9...

Searched 740 parameter combinations

series_9 optimal strategy:
  Strategy type: ma_crossover
  Parameters: fast=15,slow=30,atr_p=20,atr_m=1.5
  Training return: 10.68%
  Test return: 7.84%
  Test Sharpe: 1.62
  Test MaxDD: -2.37%
  Composite score: 0.5330

Top 5 parameter combinations:


,strategy,params,train_return,train_sharpe,valid_return,valid_sharpe,valid_maxdd,score
460,ma_crossover,"fast=15,slow=30,atr_p=20,atr_m=1.5",10.679518,0.559357,7.842834,1.619139,-2.368866,0.532985
451,ma_crossover,"fast=15,slow=30,atr_p=10,atr_m=2.0",12.861730,0.606123,6.855272,1.399329,-2.368866,0.505053
535,ma_crossover,"fast=20,slow=30,atr_p=20,atr_m=1.5",12.648792,0.659781,6.488529,1.358678,-2.874426,0.504248
525,ma_crossover,"fast=20,slow=30,atr_p=10,atr_m=1.5",12.362533,0.645754,6.488529,1.358678,-2.874426,0.502144
455,ma_crossover,"fast=15,slow=30,atr_p=14,atr_m=1.5",5.898376,0.340986,7.842834,1.619139,-2.368866,0.500229


In [8]:
# Summarize trend strategy results
print('\n' + '='*80)
print('Trend/Momentum Strategy - Optimal Parameters Summary per Asset')
print('='*80)

trend_summary_df = pd.DataFrame(trend_summary)
display(trend_summary_df)

print(f'\nAverage training return: {trend_summary_df["train_return"].mean():.2f}%')
print(f'Average test return: {trend_summary_df["valid_return"].mean():.2f}%')

# Save
trend_summary_df.to_csv(RESULTS_PATH / 'trend_individual_summary.csv', index=False)
with open(RESULTS_PATH / 'trend_optimal_params.json', 'w') as f:
    json.dump(trend_optimal_params, f, indent=2)


Trend/Momentum Strategy - Optimal Parameters Summary per Asset


,series,strategy,params,train_return,valid_return,valid_sharpe,score
0,series_1,above_ma,"ma_period=10,confirm_days=7",39.095460,12.060110,0.685916,0.429433
1,series_3,ma_crossover,"fast=20,slow=40,atr_p=10,atr_m=2.5",11.494946,12.332610,1.821230,0.618235
2,series_4,ma_crossover,"fast=15,slow=30,atr_p=14,atr_m=1.5",22.126090,46.762280,1.940744,0.614839
3,series_6,breakout,"lookback=25,hold_days=20",26.021436,23.440454,1.288207,0.484959
4,series_9,ma_crossover,"fast=15,slow=30,atr_p=20,atr_m=1.5",10.679518,7.842834,1.619139,0.532985



Average training return: 21.88%
Average test return: 20.49%


---
# Part 2: Mean Reversion Strategy Parameter Optimization

**Independently find optimal parameters** for each mean reversion asset, searching across 4 strategy variants.

## Strategy Variants
1. **Z-score Mean Reversion** (zscore)
2. **RSI Mean Reversion** (rsi)
3. **Bollinger Band Mean Reversion** (bollinger)
4. **Dual Moving Average Deviation** (ma_deviation)

## 7. Mean Reversion Strategy Signal Generation Functions

In [9]:
# ============================================================
# Strategy 1: Z-score Mean Reversion
# ============================================================
def generate_mr_signals_zscore(df: pd.DataFrame, lookback: int,
                                entry_zscore: float, exit_zscore: float) -> pd.Series:
    """Z-score mean reversion: buy on oversold, close on reversion"""
    close = df['Close']
    
    ma = close.rolling(lookback).mean()
    std = close.rolling(lookback).std()
    zscore = (close - ma) / std
    
    signals = pd.Series(0, index=df.index)
    position = 0
    
    for i in range(lookback, len(df)):
        z = zscore.iloc[i]
        
        if position == 0:
            if z < -entry_zscore:
                signals.iloc[i] = 1
                position = 1
        elif position == 1:
            if z > exit_zscore or z > entry_zscore:
                signals.iloc[i] = -1
                position = 0
    
    return signals


# ============================================================
# Strategy 2: RSI Mean Reversion
# ============================================================
def generate_mr_signals_rsi(df: pd.DataFrame, rsi_period: int,
                            oversold: float, overbought: float) -> pd.Series:
    """RSI mean reversion: buy on oversold, close when RSI reverts to 50 or overbought"""
    close = df['Close']
    
    delta = close.diff()
    gain = delta.where(delta > 0, 0).rolling(rsi_period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(rsi_period).mean()
    rs = gain / (loss + 1e-10)
    rsi = 100 - (100 / (1 + rs))
    
    signals = pd.Series(0, index=df.index)
    position = 0
    
    for i in range(rsi_period + 1, len(df)):
        r = rsi.iloc[i]
        
        if position == 0:
            if r < oversold:
                signals.iloc[i] = 1
                position = 1
        elif position == 1:
            if r > 50 or r > overbought:
                signals.iloc[i] = -1
                position = 0
    
    return signals


# ============================================================
# Strategy 3: Bollinger Band Mean Reversion
# ============================================================
def generate_mr_signals_bollinger(df: pd.DataFrame, lookback: int,
                                   std_mult: float, exit_mult: float) -> pd.Series:
    """Bollinger band mean reversion: buy below lower band, close above middle band"""
    close = df['Close']
    
    ma = close.rolling(lookback).mean()
    std = close.rolling(lookback).std()
    lower = ma - std_mult * std
    
    signals = pd.Series(0, index=df.index)
    position = 0
    
    for i in range(lookback, len(df)):
        price = close.iloc[i]
        mid = ma.iloc[i]
        
        if position == 0:
            if price < lower.iloc[i]:
                signals.iloc[i] = 1
                position = 1
        elif position == 1:
            if price > mid:
                signals.iloc[i] = -1
                position = 0
    
    return signals


# ============================================================
# Strategy 4: Dual Moving Average Deviation Mean Reversion
# ============================================================
def generate_mr_signals_ma_deviation(df: pd.DataFrame, fast_period: int,
                                      slow_period: int, threshold: float) -> pd.Series:
    """Dual MA deviation: trade against excessive deviation of fast MA from slow MA"""
    close = df['Close']
    
    fast_ma = close.rolling(fast_period).mean()
    slow_ma = close.rolling(slow_period).mean()
    deviation = (fast_ma - slow_ma) / slow_ma * 100
    
    signals = pd.Series(0, index=df.index)
    position = 0
    
    for i in range(slow_period, len(df)):
        dev = deviation.iloc[i]
        
        if position == 0:
            if dev < -threshold:
                signals.iloc[i] = 1
                position = 1
        elif position == 1:
            if dev > 0:
                signals.iloc[i] = -1
                position = 0
    
    return signals

print('Mean reversion strategy signal functions defined: zscore, rsi, bollinger, ma_deviation')

Mean reversion strategy signal functions defined: zscore, rsi, bollinger, ma_deviation


## 8. Mean Reversion Strategy Parameter Grid Definition

In [10]:
# ============================================================
# Parameter grid definition
# ============================================================

# Strategy 1: Z-score
ZSCORE_PARAM_GRID = {
    'lookback': [5, 10, 15, 20, 25, 30, 40, 50],   # 8 values
    'entry_zscore': [1.0, 1.5, 2.0, 2.5, 3.0],     # 5 values
    'exit_zscore': [-0.5, -0.25, 0.0, 0.25, 0.5]   # 5 values
}

# Strategy 2: RSI
RSI_PARAM_GRID = {
    'rsi_period': [7, 10, 14, 20, 25],             # 5 values
    'oversold': [20, 25, 30, 35],                  # 4 values
    'overbought': [65, 70, 75, 80]                 # 4 values
}

# Strategy 3: Bollinger Band
BOLLINGER_PARAM_GRID = {
    'lookback': [10, 15, 20, 25, 30, 40],          # 6 values
    'std_mult': [1.5, 2.0, 2.5, 3.0],              # 4 values
    'exit_mult': [-0.5, 0.0, 0.5, 1.0]             # 4 values
}

# Strategy 4: Dual MA Deviation
MA_DEV_PARAM_GRID = {
    'fast_period': [5, 10, 15, 20],                # 4 values
    'slow_period': [20, 30, 40, 50, 60],           # 5 values
    'threshold': [1.0, 2.0, 3.0, 4.0, 5.0]         # 5 values
}

print('=== Mean Reversion Strategy Parameter Grid ===')
print(f'\nStrategy 1 - Z-score:')
for k, v in ZSCORE_PARAM_GRID.items():
    print(f'  {k}: {v}')
print(f'  Combinations: {len(ZSCORE_PARAM_GRID["lookback"]) * len(ZSCORE_PARAM_GRID["entry_zscore"]) * len(ZSCORE_PARAM_GRID["exit_zscore"])}')

print(f'\nStrategy 2 - RSI:')
for k, v in RSI_PARAM_GRID.items():
    print(f'  {k}: {v}')
print(f'  Combinations: {len(RSI_PARAM_GRID["rsi_period"]) * len(RSI_PARAM_GRID["oversold"]) * len(RSI_PARAM_GRID["overbought"])}')

print(f'\nStrategy 3 - Bollinger Band:')
for k, v in BOLLINGER_PARAM_GRID.items():
    print(f'  {k}: {v}')
print(f'  Combinations: {len(BOLLINGER_PARAM_GRID["lookback"]) * len(BOLLINGER_PARAM_GRID["std_mult"]) * len(BOLLINGER_PARAM_GRID["exit_mult"])}')

print(f'\nStrategy 4 - Dual MA Deviation:')
for k, v in MA_DEV_PARAM_GRID.items():
    print(f'  {k}: {v}')
print(f'  Estimated combinations: ~80 (after filtering fast>=slow)')

print(f'\nEstimated total parameter combinations per asset: ~471')

=== Mean Reversion Strategy Parameter Grid ===

Strategy 1 - Z-score:
  lookback: [5, 10, 15, 20, 25, 30, 40, 50]
  entry_zscore: [1.0, 1.5, 2.0, 2.5, 3.0]
  exit_zscore: [-0.5, -0.25, 0.0, 0.25, 0.5]
  Combinations: 200

Strategy 2 - RSI:
  rsi_period: [7, 10, 14, 20, 25]
  oversold: [20, 25, 30, 35]
  overbought: [65, 70, 75, 80]
  Combinations: 80

Strategy 3 - Bollinger Band:
  lookback: [10, 15, 20, 25, 30, 40]
  std_mult: [1.5, 2.0, 2.5, 3.0]
  exit_mult: [-0.5, 0.0, 0.5, 1.0]
  Combinations: 96

Strategy 4 - Dual MA Deviation:
  fast_period: [5, 10, 15, 20]
  slow_period: [20, 30, 40, 50, 60]
  threshold: [1.0, 2.0, 3.0, 4.0, 5.0]
  Estimated combinations: ~80 (after filtering fast>=slow)

Estimated total parameter combinations per asset: ~471


## 9. Single-Asset Mean Reversion Strategy Optimization Function

In [11]:
def optimize_mr_single_asset(df: pd.DataFrame, series_name: str, train_end: int) -> pd.DataFrame:
    """
    Search all mean reversion strategy types and parameter combinations for a single asset,
    return results sorted by score.
    """
    results = []
    
    # ============================================================
    # Strategy 1: Z-score
    # ============================================================
    for lookback, entry_z, exit_z in product(
        ZSCORE_PARAM_GRID['lookback'],
        ZSCORE_PARAM_GRID['entry_zscore'],
        ZSCORE_PARAM_GRID['exit_zscore']
    ):
        try:
            signals = generate_mr_signals_zscore(df, lookback, entry_z, exit_z)
            
            bt_train = SimpleBacktest(df.iloc[:train_end])
            train_result = bt_train.run(signals.iloc[:train_end])
            
            bt_valid = SimpleBacktest(df.iloc[train_end:])
            valid_result = bt_valid.run(signals.iloc[train_end:])
            
            score = (
                0.3 * np.clip(train_result['sharpe'], -2, 2) / 2 +
                0.3 * np.clip(valid_result['sharpe'], -2, 2) / 2 +
                0.2 * np.clip(valid_result['total_return'], -50, 100) / 100 +
                0.2 * (1 - abs(np.clip(valid_result['max_drawdown'], -50, 0)) / 50)
            )
            
            results.append({
                'strategy': 'zscore',
                'params': f'lookback={lookback},entry_z={entry_z},exit_z={exit_z}',
                'train_return': train_result['total_return'],
                'train_sharpe': train_result['sharpe'],
                'valid_return': valid_result['total_return'],
                'valid_sharpe': valid_result['sharpe'],
                'valid_maxdd': valid_result['max_drawdown'],
                'score': score
            })
        except:
            continue
    
    # ============================================================
    # Strategy 2: RSI
    # ============================================================
    for rsi_period, oversold, overbought in product(
        RSI_PARAM_GRID['rsi_period'],
        RSI_PARAM_GRID['oversold'],
        RSI_PARAM_GRID['overbought']
    ):
        try:
            signals = generate_mr_signals_rsi(df, rsi_period, oversold, overbought)
            
            bt_train = SimpleBacktest(df.iloc[:train_end])
            train_result = bt_train.run(signals.iloc[:train_end])
            
            bt_valid = SimpleBacktest(df.iloc[train_end:])
            valid_result = bt_valid.run(signals.iloc[train_end:])
            
            score = (
                0.3 * np.clip(train_result['sharpe'], -2, 2) / 2 +
                0.3 * np.clip(valid_result['sharpe'], -2, 2) / 2 +
                0.2 * np.clip(valid_result['total_return'], -50, 100) / 100 +
                0.2 * (1 - abs(np.clip(valid_result['max_drawdown'], -50, 0)) / 50)
            )
            
            results.append({
                'strategy': 'rsi',
                'params': f'rsi_period={rsi_period},oversold={oversold},overbought={overbought}',
                'train_return': train_result['total_return'],
                'train_sharpe': train_result['sharpe'],
                'valid_return': valid_result['total_return'],
                'valid_sharpe': valid_result['sharpe'],
                'valid_maxdd': valid_result['max_drawdown'],
                'score': score
            })
        except:
            continue
    
    # ============================================================
    # Strategy 3: Bollinger Band
    # ============================================================
    for lookback, std_mult, exit_mult in product(
        BOLLINGER_PARAM_GRID['lookback'],
        BOLLINGER_PARAM_GRID['std_mult'],
        BOLLINGER_PARAM_GRID['exit_mult']
    ):
        try:
            signals = generate_mr_signals_bollinger(df, lookback, std_mult, exit_mult)
            
            bt_train = SimpleBacktest(df.iloc[:train_end])
            train_result = bt_train.run(signals.iloc[:train_end])
            
            bt_valid = SimpleBacktest(df.iloc[train_end:])
            valid_result = bt_valid.run(signals.iloc[train_end:])
            
            score = (
                0.3 * np.clip(train_result['sharpe'], -2, 2) / 2 +
                0.3 * np.clip(valid_result['sharpe'], -2, 2) / 2 +
                0.2 * np.clip(valid_result['total_return'], -50, 100) / 100 +
                0.2 * (1 - abs(np.clip(valid_result['max_drawdown'], -50, 0)) / 50)
            )
            
            results.append({
                'strategy': 'bollinger',
                'params': f'lookback={lookback},std_mult={std_mult},exit_mult={exit_mult}',
                'train_return': train_result['total_return'],
                'train_sharpe': train_result['sharpe'],
                'valid_return': valid_result['total_return'],
                'valid_sharpe': valid_result['sharpe'],
                'valid_maxdd': valid_result['max_drawdown'],
                'score': score
            })
        except:
            continue
    
    # ============================================================
    # Strategy 4: Dual MA Deviation
    # ============================================================
    for fast_period, slow_period, threshold in product(
        MA_DEV_PARAM_GRID['fast_period'],
        MA_DEV_PARAM_GRID['slow_period'],
        MA_DEV_PARAM_GRID['threshold']
    ):
        if fast_period >= slow_period:
            continue
        
        try:
            signals = generate_mr_signals_ma_deviation(df, fast_period, slow_period, threshold)
            
            bt_train = SimpleBacktest(df.iloc[:train_end])
            train_result = bt_train.run(signals.iloc[:train_end])
            
            bt_valid = SimpleBacktest(df.iloc[train_end:])
            valid_result = bt_valid.run(signals.iloc[train_end:])
            
            score = (
                0.3 * np.clip(train_result['sharpe'], -2, 2) / 2 +
                0.3 * np.clip(valid_result['sharpe'], -2, 2) / 2 +
                0.2 * np.clip(valid_result['total_return'], -50, 100) / 100 +
                0.2 * (1 - abs(np.clip(valid_result['max_drawdown'], -50, 0)) / 50)
            )
            
            results.append({
                'strategy': 'ma_deviation',
                'params': f'fast={fast_period},slow={slow_period},threshold={threshold}',
                'train_return': train_result['total_return'],
                'train_sharpe': train_result['sharpe'],
                'valid_return': valid_result['total_return'],
                'valid_sharpe': valid_result['sharpe'],
                'valid_maxdd': valid_result['max_drawdown'],
                'score': score
            })
        except:
            continue
    
    # Sort and return
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('score', ascending=False)
    
    return results_df

print('Single-asset mean reversion strategy optimization function defined')

Single-asset mean reversion strategy optimization function defined


## 10. Run Mean Reversion Strategy Parameter Optimization

In [12]:
print('='*80)
print('Mean Reversion Strategy Parameter Optimization - Independent per Asset')
print('='*80)

mr_optimal_params = {}
mr_summary = []

for series_name in MR_SERIES:
    print(f'\n{"="*60}')
    print(f'Optimizing {series_name}...')
    print(f'{"="*60}')
    
    df = data[series_name]
    results_df = optimize_mr_single_asset(df, series_name, TRAIN_END)
    
    # Save full results
    results_df.to_csv(RESULTS_PATH / f'mr_{series_name}_all_params.csv', index=False)
    
    # Get best result
    best = results_df.iloc[0]
    
    print(f'\nSearched {len(results_df)} parameter combinations')
    print(f'\n{series_name} optimal strategy:')
    print(f'  Strategy type: {best["strategy"]}')
    print(f'  Parameters: {best["params"]}')
    print(f'  Training return: {best["train_return"]:.2f}%')
    print(f'  Test return: {best["valid_return"]:.2f}%')
    print(f'  Test Sharpe: {best["valid_sharpe"]:.2f}')
    print(f'  Test MaxDD: {best["valid_maxdd"]:.2f}%')
    print(f'  Composite score: {best["score"]:.4f}')
    
    # Top 5
    print(f'\nTop 5 parameter combinations:')
    display(results_df.head())
    
    # Save optimal params
    mr_optimal_params[series_name] = {
        'strategy': best['strategy'],
        'params': best['params'],
        'train_return': float(best['train_return']),
        'valid_return': float(best['valid_return']),
        'valid_sharpe': float(best['valid_sharpe']),
        'valid_maxdd': float(best['valid_maxdd'])
    }
    
    mr_summary.append({
        'series': series_name,
        'strategy': best['strategy'],
        'params': best['params'],
        'train_return': best['train_return'],
        'valid_return': best['valid_return'],
        'valid_sharpe': best['valid_sharpe'],
        'score': best['score']
    })

Mean Reversion Strategy Parameter Optimization - Independent per Asset

Optimizing series_2...

Searched 471 parameter combinations

series_2 optimal strategy:
  Strategy type: zscore
  Parameters: lookback=30,entry_z=2.5,exit_z=0.5
  Training return: 4.32%
  Test return: 14.76%
  Test Sharpe: 1.41
  Test MaxDD: -2.03%
  Composite score: 0.4823

Top 5 parameter combinations:


,strategy,params,train_return,train_sharpe,valid_return,valid_sharpe,valid_maxdd,score
144,zscore,"lookback=30,entry_z=2.5,exit_z=0.5",4.31700,0.324626,14.75825,1.414311,-2.026148,0.482253
143,zscore,"lookback=30,entry_z=2.5,exit_z=0.25",4.94425,0.384435,11.33275,1.309777,-2.026148,0.468693
355,bollinger,"lookback=30,std_mult=2.5,exit_mult=1.0",6.26700,0.570375,4.25100,1.041438,-2.028306,0.442161
353,bollinger,"lookback=30,std_mult=2.5,exit_mult=0.0",6.26700,0.570375,4.25100,1.041438,-2.028306,0.442161
354,bollinger,"lookback=30,std_mult=2.5,exit_mult=0.5",6.26700,0.570375,4.25100,1.041438,-2.028306,0.442161



Optimizing series_7...

Searched 471 parameter combinations

series_7 optimal strategy:
  Strategy type: ma_deviation
  Parameters: fast=10,slow=60,threshold=1.0
  Training return: 1.22%
  Test return: 1.19%
  Test Sharpe: 1.52
  Test MaxDD: -0.44%
  Composite score: 0.6912

Top 5 parameter combinations:


,strategy,params,train_return,train_sharpe,valid_return,valid_sharpe,valid_maxdd,score
421,ma_deviation,"fast=10,slow=60,threshold=1.0",1.220617,1.751944,1.187812,1.518659,-0.441882,0.691199
320,bollinger,"lookback=20,std_mult=2.5,exit_mult=-0.5",3.935359,2.287003,0.214425,0.213194,-1.136904,0.527860
321,bollinger,"lookback=20,std_mult=2.5,exit_mult=0.0",3.935359,2.287003,0.214425,0.213194,-1.136904,0.527860
322,bollinger,"lookback=20,std_mult=2.5,exit_mult=0.5",3.935359,2.287003,0.214425,0.213194,-1.136904,0.527860
323,bollinger,"lookback=20,std_mult=2.5,exit_mult=1.0",3.935359,2.287003,0.214425,0.213194,-1.136904,0.527860



Optimizing series_8...

Searched 471 parameter combinations

series_8 optimal strategy:
  Strategy type: ma_deviation
  Parameters: fast=5,slow=50,threshold=2.0
  Training return: 7.01%
  Test return: 1.99%
  Test Sharpe: 1.47
  Test MaxDD: -0.88%
  Composite score: 0.5300

Top 5 parameter combinations:


,strategy,params,train_return,train_sharpe,valid_return,valid_sharpe,valid_maxdd,score
392,ma_deviation,"fast=5,slow=50,threshold=2.0",7.010862,0.722518,1.989250,1.474394,-0.882379,0.529986
94,zscore,"lookback=20,entry_z=2.5,exit_z=0.5",4.719384,0.574031,3.974161,1.535726,-1.709399,0.517574
193,zscore,"lookback=50,entry_z=2.5,exit_z=0.25",6.605157,0.679544,2.922004,1.363485,-1.709346,0.505461
93,zscore,"lookback=20,entry_z=2.5,exit_z=0.25",4.305489,0.527794,3.367098,1.365238,-1.709765,0.483850
194,zscore,"lookback=50,entry_z=2.5,exit_z=0.5",7.422543,0.756322,2.558461,1.051952,-1.844663,0.468979



Optimizing series_10...

Searched 471 parameter combinations

series_10 optimal strategy:
  Strategy type: zscore
  Parameters: lookback=30,entry_z=1.5,exit_z=0.25
  Training return: 41.47%
  Test return: 41.91%
  Test Sharpe: 1.68
  Test MaxDD: -16.66%
  Composite score: 0.6260

Top 5 parameter combinations:


,strategy,params,train_return,train_sharpe,valid_return,valid_sharpe,valid_maxdd,score
133,zscore,"lookback=30,entry_z=1.5,exit_z=0.25",41.46775,1.048505,41.91275,1.677013,-16.664459,0.625995
361,bollinger,"lookback=40,std_mult=1.5,exit_mult=0.0",27.15325,0.752339,41.64375,1.717354,-16.647192,0.587153
363,bollinger,"lookback=40,std_mult=1.5,exit_mult=1.0",27.15325,0.752339,41.64375,1.717354,-16.647192,0.587153
362,bollinger,"lookback=40,std_mult=1.5,exit_mult=0.5",27.15325,0.752339,41.64375,1.717354,-16.647192,0.587153
157,zscore,"lookback=40,entry_z=1.5,exit_z=0.0",27.15325,0.752339,41.64375,1.717354,-16.647192,0.587153


In [13]:
# Summarize mean reversion strategy results
print('\n' + '='*80)
print('Mean Reversion Strategy - Optimal Parameters Summary per Asset')
print('='*80)

mr_summary_df = pd.DataFrame(mr_summary)
display(mr_summary_df)

print(f'\nAverage training return: {mr_summary_df["train_return"].mean():.2f}%')
print(f'Average test return: {mr_summary_df["valid_return"].mean():.2f}%')

# Save
mr_summary_df.to_csv(RESULTS_PATH / 'mr_individual_summary.csv', index=False)
with open(RESULTS_PATH / 'mr_optimal_params.json', 'w') as f:
    json.dump(mr_optimal_params, f, indent=2)


Mean Reversion Strategy - Optimal Parameters Summary per Asset


,series,strategy,params,train_return,valid_return,valid_sharpe,score
0,series_2,zscore,"lookback=30,entry_z=2.5,exit_z=0.5",4.317000,14.758250,1.414311,0.482253
1,series_7,ma_deviation,"fast=10,slow=60,threshold=1.0",1.220617,1.187812,1.518659,0.691199
2,series_8,ma_deviation,"fast=5,slow=50,threshold=2.0",7.010862,1.989250,1.474394,0.529986
3,series_10,zscore,"lookback=30,entry_z=1.5,exit_z=0.25",41.467750,41.912750,1.677013,0.625995



Average training return: 13.50%
Average test return: 14.96%


---
# Part 3: Short/Defensive Strategy Parameter Optimization

In [14]:
def generate_short_signals(df: pd.DataFrame, sma_period: int, stop_loss_pct: float) -> pd.Series:
    """Trend short strategy: short when price drops below MA, close when price returns above MA or stop loss"""
    close = df['Close']
    sma = close.rolling(sma_period).mean()
    
    signals = pd.Series(0, index=df.index)
    position = 0
    entry_price = 0
    
    for i in range(sma_period, len(df)):
        price = close.iloc[i]
        
        if position == 0:
            if price < sma.iloc[i] and close.iloc[i-1] >= sma.iloc[i-1]:
                signals.iloc[i] = -2  # Short signal
                position = -1
                entry_price = price
        elif position == -1:
            if price > sma.iloc[i]:
                signals.iloc[i] = 1  # Close position
                position = 0
            elif price > entry_price * (1 + stop_loss_pct):
                signals.iloc[i] = 1  # Stop loss
                position = 0
    
    return signals

# Parameter grid
SHORT_PARAM_GRID = {
    'sma_period': [10, 15, 20, 25, 30, 40, 50, 100, 200],
    'stop_loss_pct': [0.02, 0.03, 0.05, 0.07, 0.10]
}

print('='*80)
print('Short/Defensive Strategy Parameter Optimization')
print('='*80)
print(f'Short assets: {SHORT_SERIES}')
print(f'\nParameter grid:')
for k, v in SHORT_PARAM_GRID.items():
    print(f'  {k}: {v}')
print(f'\nTotal parameter combinations: {len(SHORT_PARAM_GRID["sma_period"]) * len(SHORT_PARAM_GRID["stop_loss_pct"])}')

Short/Defensive Strategy Parameter Optimization
Short assets: ['series_5']

Parameter grid:
  sma_period: [10, 15, 20, 25, 30, 40, 50, 100, 200]
  stop_loss_pct: [0.02, 0.03, 0.05, 0.07, 0.1]

Total parameter combinations: 45


In [15]:
short_results = []

for sma_period, stop_loss in product(
    SHORT_PARAM_GRID['sma_period'],
    SHORT_PARAM_GRID['stop_loss_pct']
):
    df = data['series_5']
    
    try:
        signals = generate_short_signals(df, sma_period, stop_loss)
        
        bt_train = SimpleBacktest(df.iloc[:TRAIN_END])
        train_result = bt_train.run(signals.iloc[:TRAIN_END])
        
        bt_valid = SimpleBacktest(df.iloc[TRAIN_END:])
        valid_result = bt_valid.run(signals.iloc[TRAIN_END:])
        
        score = (
            0.3 * np.clip(train_result['sharpe'], -2, 2) / 2 +
            0.3 * np.clip(valid_result['sharpe'], -2, 2) / 2 +
            0.2 * np.clip(valid_result['total_return'], -50, 100) / 100 +
            0.2 * (1 - abs(np.clip(valid_result['max_drawdown'], -50, 0)) / 50)
        )
        
        short_results.append({
            'sma_period': sma_period,
            'stop_loss_pct': stop_loss,
            'train_return': train_result['total_return'],
            'train_sharpe': train_result['sharpe'],
            'valid_return': valid_result['total_return'],
            'valid_sharpe': valid_result['sharpe'],
            'valid_maxdd': valid_result['max_drawdown'],
            'score': score
        })
    except:
        continue

short_df = pd.DataFrame(short_results).sort_values('score', ascending=False)
short_df.to_csv(RESULTS_PATH / 'short_all_params.csv', index=False)

print(f'\nSearched {len(short_df)} parameter combinations')
print('\nTop 10 parameter combinations:')
display(short_df.head(10))

best_short = short_df.iloc[0]
print(f'\nOptimal parameters: sma_period={int(best_short["sma_period"])}, stop_loss_pct={best_short["stop_loss_pct"]}')
print(f'Test return: {best_short["valid_return"]:.2f}%')


Searched 45 parameter combinations

Top 10 parameter combinations:


,sma_period,stop_loss_pct,train_return,train_sharpe,valid_return,valid_sharpe,valid_maxdd,score
41,200,0.03,20.89510,1.245952,3.06360,0.791117,-2.160167,0.503047
40,200,0.02,20.89510,1.245952,3.06360,0.791117,-2.160167,0.503047
42,200,0.05,20.89510,1.245952,3.06360,0.791117,-2.160167,0.503047
43,200,0.07,20.89510,1.245952,3.06360,0.791117,-2.160167,0.503047
44,200,0.10,20.89510,1.245952,3.06360,0.791117,-2.160167,0.503047
26,40,0.03,16.76609,0.817001,2.38795,0.293214,-8.197215,0.338519
27,40,0.05,16.76609,0.817001,2.38795,0.293214,-8.197215,0.338519
25,40,0.02,16.76609,0.817001,2.38795,0.293214,-8.197215,0.338519
29,40,0.10,16.76609,0.817001,2.38795,0.293214,-8.197215,0.338519
28,40,0.07,16.76609,0.817001,2.38795,0.293214,-8.197215,0.338519



Optimal parameters: sma_period=200, stop_loss_pct=0.03
Test return: 3.06%


---
# Summary: Optimal Parameters Overview

In [16]:
print('='*80)
print('Optimal Parameters Summary for All Three Strategy Types - Independent per Asset')
print('='*80)

print('\n[Trend/Momentum Strategy]')
print('-'*60)
for series_name, params in trend_optimal_params.items():
    print(f'{series_name}:')
    print(f'  Strategy: {params["strategy"]}')
    print(f'  Parameters: {params["params"]}')
    print(f'  Test return: {params["valid_return"]:.2f}%, Sharpe: {params["valid_sharpe"]:.2f}')

print(f'\n[Mean Reversion Strategy]')
print('-'*60)
for series_name, params in mr_optimal_params.items():
    print(f'{series_name}:')
    print(f'  Strategy: {params["strategy"]}')
    print(f'  Parameters: {params["params"]}')
    print(f'  Test return: {params["valid_return"]:.2f}%, Sharpe: {params["valid_sharpe"]:.2f}')

print(f'\n[Short/Defensive Strategy]')
print('-'*60)
print(f'series_5:')
print(f'  Parameters: sma_period={int(best_short["sma_period"])}, stop_loss_pct={best_short["stop_loss_pct"]}')
print(f'  Test return: {best_short["valid_return"]:.2f}%, Sharpe: {best_short["valid_sharpe"]:.2f}')

Optimal Parameters Summary for All Three Strategy Types - Independent per Asset

[Trend/Momentum Strategy]
------------------------------------------------------------
series_1:
  Strategy: above_ma
  Parameters: ma_period=10,confirm_days=7
  Test return: 12.06%, Sharpe: 0.69
series_3:
  Strategy: ma_crossover
  Parameters: fast=20,slow=40,atr_p=10,atr_m=2.5
  Test return: 12.33%, Sharpe: 1.82
series_4:
  Strategy: ma_crossover
  Parameters: fast=15,slow=30,atr_p=14,atr_m=1.5
  Test return: 46.76%, Sharpe: 1.94
series_6:
  Strategy: breakout
  Parameters: lookback=25,hold_days=20
  Test return: 23.44%, Sharpe: 1.29
series_9:
  Strategy: ma_crossover
  Parameters: fast=15,slow=30,atr_p=20,atr_m=1.5
  Test return: 7.84%, Sharpe: 1.62

[Mean Reversion Strategy]
------------------------------------------------------------
series_2:
  Strategy: zscore
  Parameters: lookback=30,entry_z=2.5,exit_z=0.5
  Test return: 14.76%, Sharpe: 1.41
series_7:
  Strategy: ma_deviation
  Parameters: fast=10

In [17]:
# Save all optimal parameters
all_optimal_params = {
    'trend': trend_optimal_params,
    'mean_reversion': mr_optimal_params,
    'short': {
        'series_5': {
            'sma_period': int(best_short['sma_period']),
            'stop_loss_pct': float(best_short['stop_loss_pct']),
            'valid_return': float(best_short['valid_return']),
            'valid_sharpe': float(best_short['valid_sharpe'])
        }
    }
}

with open(RESULTS_PATH / 'all_optimal_params.json', 'w') as f:
    json.dump(all_optimal_params, f, indent=2, default=str)

print(f'All optimal parameters saved to: {RESULTS_PATH / "all_optimal_params.json"}')

All optimal parameters saved to: optimization_results\all_optimal_params.json


In [18]:
# Expected portfolio return estimate
print('\n' + '='*80)
print('Expected Portfolio Return Estimate')
print('='*80)

trend_avg = trend_summary_df['valid_return'].mean()
mr_avg = mr_summary_df['valid_return'].mean()
short_return = best_short['valid_return']

print(f'\nTrend strategy average test return ({len(TREND_SERIES)} assets): {trend_avg:.2f}%')
print(f'Mean reversion strategy average test return ({len(MR_SERIES)} assets): {mr_avg:.2f}%')
print(f'Short strategy test return ({len(SHORT_SERIES)} assets): {short_return:.2f}%')

# Weighted average
total_assets = len(TREND_SERIES) + len(MR_SERIES) + len(SHORT_SERIES)
weighted_avg = (trend_avg * len(TREND_SERIES) + mr_avg * len(MR_SERIES) + short_return * len(SHORT_SERIES)) / total_assets

print(f'\nPortfolio weighted average test return: {weighted_avg:.2f}%')
print(f'\nTarget: 30%')
print(f'Target met: {"Yes" if weighted_avg >= 30 else "No"}')


Expected Portfolio Return Estimate

Trend strategy average test return (5 assets): 20.49%
Mean reversion strategy average test return (4 assets): 14.96%
Short strategy test return (1 assets): 3.06%

Portfolio weighted average test return: 16.53%

Target: 30%
Target met: No
